# 01. Cleaning Validation

Goal: confirm that `src/clean.py` fixed every issue found in `00_raw_data_audit` and did not introduce new ones.
The cleaning logic lives only in `src/clean.py`. This notebook imports those functions so the checks always match the real pipeline.

**Sections**
1. Before and after, step by step
2. Sanity checks
3. Assumptions and decisions

Cleaning is a loop. If a check here fails, fix `clean.py`, rerun `python src/clean.py`, and rerun this notebook.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# The notebook lives in notebooks/, so the repository root is one level up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import clean  # src/clean.py

RAW_PATH = REPO_ROOT / "data" / "raw" / "messi_all_goals.csv"
CLEAN_PATH = REPO_ROOT / "data" / "clean" / "messi_goals_clean.csv"

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

raw = clean.load_raw_data(RAW_PATH)
print(f"Raw rows: {len(raw)}")

Raw rows: 790


## 1. Before and after, step by step

Each cleaning function is applied in order to a copy of the raw data. After each step, a few rows show what changed.

In [2]:
work = raw.copy()

**Step 1: dates** (text such as `9/5/2004` becomes a real date)

In [3]:
work = clean.clean_dates(work)
print(work["date"].dtype)
work[["date", "year", "month"]].head(3)

datetime64[ns]


,date,year,month
0,2004-09-05,2004,9
1,2004-09-18,2004,9
2,2004-10-31,2004,10


**Step 2: stage** (`match_stage` is split into `stage_name` and `matchday`)

In [4]:
work = clean.clean_stage(work)
work[["competition", "stage_name", "matchday"]].drop_duplicates().sort_values(["competition", "stage_name"]).head(12)

,competition,stage_name,matchday
0,2ª B - Grupo III,League,2
1,2ª B - Grupo III,League,4
2,2ª B - Grupo III,League,10
3,2ª B - Grupo III,League,14
4,2ª B - Grupo III,League,17
5,2ª B - Grupo III,League,33
744,CONCACAF Champions Cup,First Round,<NA>
750,CONCACAF Champions Cup,Quarter-Finals,<NA>
724,CONCACAF Champions Cup,Round of 16,<NA>
101,Club World Cup,Final,<NA>


**Step 3: scores** (parsed into numbers, then flipped to Messi's team point of view using `venue`)

Away games are the best check: the raw score is home:away, so the numbers must be swapped for Messi's team.

In [5]:
before_scores = work[["venue", "match_score", "score_at_goal"]].copy()
work = clean.clean_scores(work)

comparison = pd.concat(
    [
        before_scores,
        work[
            [
                "team_goals_final", "opponent_goals_final", "match_result",
                "team_goals_before", "opponent_goals_before", "game_state_before",
            ]
        ],
    ],
    axis=1,
)
comparison[comparison["venue"] == "Away"].head(5)

,venue,match_score,score_at_goal,team_goals_final,opponent_goals_final,match_result,team_goals_before,opponent_goals_before,game_state_before
2,Away,2:01,0:01,1,2,L,0,0,Drawing
3,Away,2:02,0:01,2,2,D,0,0,Drawing
11,Away,0:03,0:02,3,0,W,1,0,Winning
12,Away,0:03,0:03,3,0,W,2,0,Winning
15,Away,2:03,1:02,3,2,W,1,1,Drawing


**Step 4: assists** (`Not Applicable` becomes `has_assist = False`)

In [6]:
work = clean.clean_assists(work)
work[["goal_type", "assist_player", "has_assist"]].head(5)

,goal_type,assist_player,has_assist
0,Left-footed shot,Dani Fernández,True
1,Left-footed shot,Dani Fernández,True
2,Not reported,<NA>,False
3,Not reported,<NA>,False
4,Left-footed shot,<NA>,False


**Step 5: goal types** (13 raw types grouped, raw value kept in `goal_type_raw`)

In [7]:
work = clean.clean_goal_types(work)
pd.crosstab(work["goal_type_raw"], work["goal_type"])

goal_type,Free kick,Header,Left foot,Other,Penalty,Right foot,Unknown
goal_type_raw,,,,,,,
Chest,0,0,0,2,0,0,0
Counter attack goal,0,0,0,1,0,0,0
Deflected shot on goal,0,0,0,2,0,0,0
Direct free kick,59,0,0,0,0,0,0
Header,0,28,0,0,0,0,0
Left-footed shot,0,0,493,0,0,0,0
Long distance kick,0,0,0,1,0,0,0
Not reported,0,0,0,0,0,0,5
Penalty,0,0,0,0,88,0,0


**Step 6: flags, minute bucket order, match id**

In [8]:
work = clean.clean_flags_and_categories(work)
work = clean.add_match_id(work)
work = clean.reorder_columns(work)

print(f"Matches (match_id): {work['match_id'].nunique()}")
print(f"Rows: {len(work)}, columns: {work.shape[1]}")
work.head(3)

Matches (match_id): 518
Rows: 790, columns: 30


,match_id,goal_number_in_match,date,year,month,season,goal_decade,club,competition,stage_name,matchday,opponent,venue,is_home_goal,player_position,goal_minute,goal_minute_bucket,goal_type,goal_type_raw,assist_player,has_assist,team_goals_before,opponent_goals_before,score_diff_before,game_state_before,team_goals_after,opponent_goals_after,team_goals_final,opponent_goals_final,match_result
0,1,1,2004-09-05,2004,9,2004-05,2000s,FC Barcelona B,2ª B - Grupo III,League,2,Girona FC,Home,True,LW,65,61-75,Left foot,Left-footed shot,Dani Fernández,True,0,2,-2,Losing,1,2,1,2,L
1,2,1,2004-09-18,2004,9,2004-05,2000s,FC Barcelona B,2ª B - Grupo III,League,4,RCD Espanyol B,Home,True,LW,88,76-90,Left foot,Left-footed shot,Dani Fernández,True,1,0,1,Winning,2,0,2,0,W
2,3,1,2004-10-31,2004,10,2004-05,2000s,FC Barcelona B,2ª B - Grupo III,League,10,Villajoyosa CF,Away,False,AM,2,0-15,Unknown,Not reported,<NA>,False,0,0,0,Drawing,1,0,1,2,L


**Consistency check.** The step-by-step result should be identical to the file that `python src/clean.py` saved. If this fails, the saved file is out of date and `clean.py` needs to be rerun.

In [9]:
saved = pd.read_csv(CLEAN_PATH, parse_dates=["date"])


def normalize_for_comparison(frame):
    # A CSV round trip turns the integer matchday into a float and prints blanks
    # differently, so convert both tables to the same plain-text form first.
    frame = frame.copy()
    frame["matchday"] = pd.to_numeric(frame["matchday"]).astype("float64")
    frame = frame.astype(object).where(frame.notna(), "")
    return frame.astype(str).reset_index(drop=True)


same_shape = work.shape == saved.shape
same_values = same_shape and normalize_for_comparison(work).equals(
    normalize_for_comparison(saved)
)
print(f"Same shape: {same_shape}")
print(f"Same values as saved clean file: {same_values}")

Same shape: True
Same values as saved clean file: True


## 2. Sanity checks

These checks must all pass. They catch the most likely mistakes in the score logic.

In [10]:
clean_df = work

checks = {
    "Row count unchanged": len(clean_df) == len(raw),
    "No goals-after above final goals (team)": (
        clean_df["team_goals_after"] <= clean_df["team_goals_final"]
    ).all(),
    "No goals-after above final goals (opponent)": (
        clean_df["opponent_goals_after"] <= clean_df["opponent_goals_final"]
    ).all(),
    "No negative goals before": (clean_df["team_goals_before"] >= 0).all(),
    "is_home_goal agrees with venue": (
        clean_df["is_home_goal"] == (clean_df["venue"] == "Home")
    ).all(),
    "Messi goals per match <= team final goals": (
        clean_df.groupby("match_id").size()
        <= clean_df.groupby("match_id")["team_goals_final"].first()
    ).all(),
    "League rows always have a matchday": (
        clean_df.loc[clean_df["stage_name"] == "League", "matchday"].notna().all()
    ),
    "Cup rows never have a matchday": (
        clean_df.loc[clean_df["stage_name"] != "League", "matchday"].isna().all()
    ),
    "has_assist agrees with assist_player": (
        clean_df["has_assist"] == clean_df["assist_player"].notna()
    ).all(),
}

results = pd.Series(checks, name="passed")
print(results.to_string())
assert results.all(), "At least one sanity check failed. See the table above."

Row count unchanged                            True
No goals-after above final goals (team)        True
No goals-after above final goals (opponent)    True
No negative goals before                       True
is_home_goal agrees with venue                 True
Messi goals per match <= team final goals      True
League rows always have a matchday             True
Cup rows never have a matchday                 True
has_assist agrees with assist_player           True


## 3. Assumptions and decisions

These go into the Methods section of the report.

| Decision | Assumption or rule | How it is handled | To verify |
|---|---|---|---|
| Score format | Scores are home:away, so `1:02` means home 1, away 2 | Parsed in `clean.py`, then flipped to Messi's team using `venue` | Check 2 or 3 known matches against an outside source (see below) |
| Assists | `Not Applicable` means no assist | `has_assist = False`, `assist_player` left blank | None |
| Goal types | 13 raw types are grouped into 7 | New column `goal_type`; original kept as `goal_type_raw` | Agree on the grouping as a team |
| Missing goal type | `Not reported` is genuinely missing | 5 rows labeled `Unknown` | None |
| Scope | The data covers club goals only (Barcelona, PSG, Inter Miami) | State this limit in the report | None |
| Stoppage time | No separate stoppage-time field | Bucket `91+` mixes stoppage time and extra time | None |

### Manual check of the score assumption

Pick a few well-known matches and compare them with an outside source. Below are the matches with the most Messi goals, which are the easiest to recognize.

In [11]:
top_matches = (
    clean_df.groupby("match_id")
    .agg(
        date=("date", "first"),
        club=("club", "first"),
        competition=("competition", "first"),
        opponent=("opponent", "first"),
        venue=("venue", "first"),
        messi_goals=("goal_number_in_match", "max"),
        final_score_for_team=("team_goals_final", "first"),
        final_score_against=("opponent_goals_final", "first"),
    )
    .sort_values(["messi_goals", "date"], ascending=[False, True])
    .head(8)
)
top_matches

,date,club,competition,opponent,venue,messi_goals,final_score_for_team,final_score_against
match_id,,,,,,,,
152,2012-03-07,FC Barcelona,UEFA Champions League,Bayer 04 Leverkusen,Home,5,7,1
90,2010-04-06,FC Barcelona,UEFA Champions League,Arsenal FC,Home,4,4,1
150,2012-02-19,FC Barcelona,LaLiga,Valencia CF,Home,4,5,1
164,2012-05-05,FC Barcelona,LaLiga,RCD Espanyol Barcelona,Home,4,4,0
190,2013-01-27,FC Barcelona,LaLiga,CA Osasuna,Home,4,5,1
332,2017-09-19,FC Barcelona,LaLiga,SD Eibar,Home,4,6,1
402,2020-02-22,FC Barcelona,LaLiga,SD Eibar,Home,4,5,0
19,2007-03-10,FC Barcelona,LaLiga,Real Madrid,Home,3,3,3


If any of these rows do not match the real result, the home:away assumption is wrong. Fix it once in `parse_score_text` and `clean_scores` in `src/clean.py`, rerun `python src/clean.py`, and rerun this notebook.